In [2]:
# Load in packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import scipy.stats as stats
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

In [3]:
path = "/project/PlatigLab/users/yogi/ENCODE-RNA-Binding-Protein-Network-Modeling/analysis/06_first_order_SHAP_analysis/outputs/dpsi_vs_local_SHAP/all_partitions/dpsi_vs_local_SHAP_scatterplot_data_all_partitions.tsv.gz"

In [4]:
table = pd.read_csv(path, sep='\t', compression='gzip')

In [44]:
print(len(table))

256693


In [30]:
# Check to see how many instances d Local SHAP = 0 but significant
new_df = table[
    (table['dPSI'].abs() >= 0.1) & 
    (table['rMATS FDR'] <= 0.1) & 
    (table['CTRL - KD Local SHAP'] == 0)
].copy()

In [14]:
# Set significance threshold

# Create sig_df with significant events
sig_df = table[
    (table['dPSI'].abs() >= 0.05) & 
    (table['rMATS FDR'] <= 0.1)
].copy()

In [15]:
# Make a per feature loop that calculates a FET for each sign dPSI and d Local SHAP

def fishers_exact_test(cell_line):
    
    # Subset for cell line
    subset = sig_df[sig_df['Cell Line'] == cell_line].copy()
    
    # Get unique features
    unique_features = subset['Feature'].unique()
    
    # List to store results
    results = []
    
    # Loop through each feature
    for feature in unique_features:
        
        # Filter for this feature
        feature_subset = subset[subset['Feature'] == feature].copy()
        
        # Ignore rows where CTRL - KD Local SHAP = 0
        feature_subset = feature_subset[feature_subset['CTRL - KD Local SHAP'] != 0].copy()
        
        # Skip if no rows remain
        if len(feature_subset) == 0:
            continue
        
        # Create binary variables for signs
        feature_subset['dPSI_positive'] = (feature_subset['dPSI'] > 0).astype(int)
        feature_subset['SHAP_positive'] = (feature_subset['CTRL - KD Local SHAP'] > 0).astype(int)
        
        # Check if we have variation in both variables
        if feature_subset['dPSI_positive'].nunique() < 2 or feature_subset['SHAP_positive'].nunique() < 2:
            # Can't perform Fisher's test - set as NA
            feature_subset['Odds Ratio'] = float('nan')
            feature_subset['p-Value'] = float('nan')
        else:
            # Create 2x2 contingency table
            contingency_table = pd.crosstab(
                feature_subset['dPSI_positive'], 
                feature_subset['SHAP_positive']
            )
            
            # Perform Fisher's exact test (two-tailed)
            oddsratio, p_value = stats.fisher_exact(contingency_table, alternative='two-sided')
            
            # Add odds ratio and p-value to the feature subset
            feature_subset['Odds Ratio'] = oddsratio
            feature_subset['p-Value'] = p_value
        
        # Append to results
        results.append(feature_subset)
    
    # Combine all results into a single dataframe
    results_df = pd.concat(results, ignore_index=True)
    return results_df

In [16]:
HepG2 = fishers_exact_test("HepG2")
HepG2.to_csv('2_HepG2_FET_results.csv', index = False)

In [17]:
K562 = fishers_exact_test("K562")
K562.to_csv('2_K562_FET_results.csv', index = False)

In [41]:
# Make a per feature loop that calculates a FET for each sign dPSI and d Local SHAP

def fishers_exact_test(cell_line):
    
    # Subset for cell line
    subset = sig_df[sig_df['Cell Line'] == cell_line].copy()

    # Get unique features
    unique_features = subset['Feature'].unique()
    
    # List to store results
    results = []
    
    # Loop through each feature
    for feature in unique_features:
        
        # Filter for this feature
        feature_subset = subset[subset['Feature'] == feature].copy()
        
        # Ignore rows where CTRL - KD Local SHAP = 0
        feature_subset = feature_subset[feature_subset['CTRL - KD Local SHAP'] != 0].copy()
        
        # Skip if no rows remain
        if len(feature_subset) == 0:
            continue
        
        # Create binary variables for signs
        feature_subset['dPSI_positive'] = (feature_subset['dPSI'] > 0).astype(int)
        feature_subset['SHAP_positive'] = (feature_subset['CTRL - KD Local SHAP'] > 0).astype(int)
        
        # Check if for variation in both variables
        if feature_subset['dPSI_positive'].nunique() < 2 or feature_subset['SHAP_positive'].nunique() < 2:
            # Can't perform Fisher's test - set as NA
            feature_subset['Odds Ratio'] = float('nan')
            feature_subset['p-Value'] = float('nan')
        else:
            # Create 2x2 contingency table
            contingency_table = pd.crosstab(
                feature_subset['dPSI_positive'], 
                feature_subset['SHAP_positive']
            )
            
            # Perform Fisher's exact test (two-tailed)
            oddsratio, p_value = stats.fisher_exact(contingency_table, alternative='two-sided')
            
            # Add odds ratio and p-value to the feature subset
            feature_subset['Odds Ratio'] = oddsratio
            feature_subset['p-Value'] = p_value
        
        # Append to results
        results.append(feature_subset)
    
    # Combine all results into a single dataframe
    results_df = pd.concat(results, ignore_index=True)
    
    # Create summary dataframe with unique features and their statistics
    summary_df = results_df[['Feature', 'Odds Ratio', 'p-Value']].drop_duplicates().reset_index(drop=True)

    return summary_df

In [42]:
HepG2 = fishers_exact_test("HepG2")
HepG2 = HepG2.dropna(subset=['p-Value'])
HepG2.to_csv('2_HepG2_FET_summary.csv', index = False)

In [45]:
K562 = fishers_exact_test("K562")
K562 = K562.dropna(subset=['p-Value'])
K562.to_csv('K562_FET_summary.csv', index = False)

In [19]:
# For testing

from scipy.stats import fisher_exact

# Construct contingency table
table = [[3, 6],
         [1, 0]]

oddsratio, p_value = fisher_exact(table, alternative='two-sided')
print("Odds Ratio:", oddsratio)
print("Two-sided p-value:", round(p_value, 2))

# One-sided test (treatment better)
_, p_value_one = fisher_exact(table, alternative='greater')
print("One-sided p-value:", round(p_value_one, 2))

Odds Ratio: 0.0
Two-sided p-value: 0.4
One-sided p-value: 1.0
